# 02 — Full-Season Event Ingestion

## Purpose

This notebook scales the event-ingestion proof of concept developed in Phase 1 from one sample match to the complete selected season.

The Phase 1 notebook established the validated match-level infrastructure:

- competition and season discovery;
- complete match ingestion;
- bookmaker-data integration;
- de-vigged market probabilities;
- chronological train/test partitioning;
- target construction;
- leakage-safe rolling match features;
- one-match event and in-play snapshot validation.

The purpose of this notebook is to retrieve, validate, cache and export the complete StatsBomb event stream for every match in the selected La Liga 2015/16 season.

The resulting full-season event dataset will support:

1. richer historical pre-match features;
2. the multilayer-network paper reimplementation;
3. player- and lineup-derived features;
4. full in-play snapshot generation;
5. later classification and regression modelling.

## Central design principle

Each event must remain linked to its parent `match_id`.

This relational link allows the event data to inherit:

- match date;
- home and away team identities;
- chronological train/test assignment;
- final targets;
- bookmaker benchmark information.

The notebook also uses local caching so that successfully downloaded event files do not need to be requested repeatedly.

In [1]:
# -------------------------------------------------------------------------
# IMPORTS
# -------------------------------------------------------------------------

from pathlib import Path
import json
import time
import random
import warnings

import numpy as np
import pandas as pd

from statsbombpy import sb


# -------------------------------------------------------------------------
# CONFIGURATION
# -------------------------------------------------------------------------

CONFIG = {
    "SEED": 42,

    # Phase 1 processed match table
    "MATCH_INPUT_PATH": (
        "../data/processed/"
        "integrated_matches_laliga_2015_2016.parquet"
    ),

    # Full concatenated event output
    "EVENT_OUTPUT_PATH": (
        "../data/processed/"
        "events_laliga_2015_2016.parquet"
    ),

    # Per-match cache directory
    "EVENT_CACHE_DIR": (
        "../data/cache/events_laliga_2015_2016"
    ),

    # Ingestion log
    "INGESTION_LOG_PATH": (
        "../data/processed/"
        "event_ingestion_log_laliga_2015_2016.csv"
    ),

    # Request pacing
    "REQUEST_DELAY_SECONDS": 0.10,

    # Number of retries after an unsuccessful request
    "MAX_RETRIES": 3,

    # Initially keep False.
    # Change to True only when intentionally replacing cached files.
    "OVERWRITE_CACHE": False,
}


# -------------------------------------------------------------------------
# REPRODUCIBILITY AND DISPLAY
# -------------------------------------------------------------------------

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Full-season event-ingestion environment initialized.")
print(f"Random seed: {CONFIG['SEED']}")

Full-season event-ingestion environment initialized.
Random seed: 42


In [2]:
# -------------------------------------------------------------------------
# PROJECT PATHS
# -------------------------------------------------------------------------

MATCH_INPUT_PATH = Path(CONFIG["MATCH_INPUT_PATH"])
EVENT_OUTPUT_PATH = Path(CONFIG["EVENT_OUTPUT_PATH"])
EVENT_CACHE_DIR = Path(CONFIG["EVENT_CACHE_DIR"])
INGESTION_LOG_PATH = Path(CONFIG["INGESTION_LOG_PATH"])

EVENT_OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

EVENT_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

INGESTION_LOG_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

if not MATCH_INPUT_PATH.exists():
    raise FileNotFoundError(
        "The Phase 1 match table was not found.\n"
        f"Expected path: {MATCH_INPUT_PATH.resolve()}\n"
        "Run the export cells at the end of "
        "01_data_integration.ipynb first."
    )

print("Project paths validated.")
print(f"Match input : {MATCH_INPUT_PATH.resolve()}")
print(f"Event cache : {EVENT_CACHE_DIR.resolve()}")
print(f"Event output: {EVENT_OUTPUT_PATH.resolve()}")
print(f"Log output  : {INGESTION_LOG_PATH.resolve()}")

Project paths validated.
Match input : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/integrated_matches_laliga_2015_2016.parquet
Event cache : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/cache/events_laliga_2015_2016
Event output: /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/events_laliga_2015_2016.parquet
Log output  : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/event_ingestion_log_laliga_2015_2016.csv


## Loading the validated Phase 1 match table

The full-season event pipeline begins from the processed match table exported by the first notebook.

This avoids rediscovering the competition and downloading the schedule again. More importantly, it ensures that the event pipeline uses exactly the same set of match identifiers as the odds integration, targets and temporal split established in Phase 1.

In [3]:
# -------------------------------------------------------------------------
# LOAD PHASE 1 MATCH TABLE
# -------------------------------------------------------------------------

matches_df = pd.read_parquet(MATCH_INPUT_PATH)

required_match_columns = {
    "match_id",
    "clean_date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
}

missing_match_columns = (
    required_match_columns - set(matches_df.columns)
)

if missing_match_columns:
    raise ValueError(
        "The Phase 1 match table is missing required columns: "
        f"{sorted(missing_match_columns)}"
    )

matches_df["clean_date"] = pd.to_datetime(
    matches_df["clean_date"],
    errors="raise"
)

matches_df = matches_df.sort_values(
    by=["clean_date", "match_id"]
).reset_index(drop=True)

assert matches_df["match_id"].notna().all(), (
    "Missing match IDs detected."
)

assert matches_df["match_id"].is_unique, (
    "Duplicate match IDs detected."
)

assert not matches_df.empty, (
    "The Phase 1 match table is empty."
)

print("Phase 1 match table loaded successfully.")
print(f"Matches   : {len(matches_df):,}")
print(
    f"Date range: "
    f"{matches_df['clean_date'].min().date()} "
    f"to {matches_df['clean_date'].max().date()}"
)
print(
    f"Unique teams: "
    f"{pd.concat([matches_df['home_team'], matches_df['away_team']]).nunique()}"
)

display(
    matches_df[
        [
            "match_id",
            "clean_date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
        ]
    ].head()
)

Phase 1 match table loaded successfully.
Matches   : 380
Date range: 2015-08-21 to 2016-05-15
Unique teams: 20


,match_id,clean_date,home_team,away_team,home_score,away_score
0,3825562,2015-08-21,Málaga,Sevilla,0,0
1,3825563,2015-08-22,Atlético Madrid,Las Palmas,1,0
2,3825564,2015-08-22,RC Deportivo La Coruña,Real Sociedad,0,0
3,3825565,2015-08-22,Espanyol,Getafe,1,0
4,3825566,2015-08-22,Rayo Vallecano,Valencia,0,0


## Single-match loader validation

Before executing the full-season loop, the event loader is tested on one fixture.

This controlled validation checks:

- whether StatsBomb returns an event table;
- whether required event fields are available;
- whether the parent match ID is attached correctly;
- whether cache writing and reloading work;
- whether temporal event columns can be preserved.

Only after this test succeeds should the ingestion function be applied to all matches.

In [4]:
# -------------------------------------------------------------------------
# SINGLE-MATCH EVENT LOADER
# -------------------------------------------------------------------------

def load_match_events(
    match_id,
    cache_dir,
    overwrite_cache=False,
    max_retries=3,
    request_delay_seconds=0.10,
):
    """
    Load one StatsBomb event stream.

    The function first checks for a local Parquet cache. If no cache
    exists, it requests the events from StatsBomb, validates the result,
    attaches the parent match ID and saves the table locally.

    Parameters
    ----------
    match_id : int
        StatsBomb match identifier.

    cache_dir : pathlib.Path
        Directory containing one Parquet file per match.

    overwrite_cache : bool
        If True, ignore an existing cache and request the data again.

    max_retries : int
        Maximum number of API attempts.

    request_delay_seconds : float
        Delay between API attempts.

    Returns
    -------
    events_df : pandas.DataFrame
        Validated event table for one match.

    source : str
        Either "cache" or "api".
    """

    match_id = int(match_id)
    cache_path = cache_dir / f"match_{match_id}.parquet"

    # ---------------------------------------------------------------------
    # LOAD EXISTING CACHE
    # ---------------------------------------------------------------------
    if cache_path.exists() and not overwrite_cache:
        cached_events_df = pd.read_parquet(cache_path)

        if cached_events_df.empty:
            raise ValueError(
                f"Cached event table is empty for match {match_id}."
            )

        if "match_id" not in cached_events_df.columns:
            raise ValueError(
                f"Cached event table for match {match_id} "
                "does not contain match_id."
            )

        if not (
            cached_events_df["match_id"].astype(int) == match_id
        ).all():
            raise ValueError(
                f"Cached event rows contain an incorrect match_id "
                f"for match {match_id}."
            )

        return cached_events_df, "cache"

    # ---------------------------------------------------------------------
    # REQUEST FROM STATSBOMB
    # ---------------------------------------------------------------------
    last_exception = None

    for attempt_number in range(1, max_retries + 1):
        try:
            events_df = sb.events(match_id=match_id)

            if not isinstance(events_df, pd.DataFrame):
                raise TypeError(
                    "StatsBomb did not return a pandas DataFrame."
                )

            if events_df.empty:
                raise ValueError(
                    f"StatsBomb returned no events for match {match_id}."
                )

            required_event_columns = {
                "id",
                "index",
                "period",
                "timestamp",
                "minute",
                "second",
                "type",
                "team",
            }

            missing_event_columns = (
                required_event_columns - set(events_df.columns)
            )

            if missing_event_columns:
                raise ValueError(
                    f"Match {match_id} is missing required event columns: "
                    f"{sorted(missing_event_columns)}"
                )

            events_df = events_df.copy()
            events_df["match_id"] = match_id

            if events_df["id"].isna().any():
                raise ValueError(
                    f"Missing event IDs detected for match {match_id}."
                )

            if events_df["index"].isna().any():
                raise ValueError(
                    f"Missing event ordering indices detected "
                    f"for match {match_id}."
                )

            events_df = events_df.sort_values(
                by=["period", "index"]
            ).reset_index(drop=True)

            # Save one cache file per match.
            events_df.to_parquet(
                cache_path,
                index=False
            )

            time.sleep(request_delay_seconds)

            return events_df, "api"

        except Exception as exc:
            last_exception = exc

            print(
                f"Attempt {attempt_number}/{max_retries} failed "
                f"for match {match_id}: {exc}"
            )

            if attempt_number < max_retries:
                time.sleep(request_delay_seconds * attempt_number)

    raise RuntimeError(
        f"Event ingestion failed for match {match_id} "
        f"after {max_retries} attempts."
    ) from last_exception

In [5]:
# -------------------------------------------------------------------------
# TEST EVENT LOADER ON ONE MATCH
# -------------------------------------------------------------------------

sample_match = matches_df.iloc[0]

SAMPLE_MATCH_ID = int(sample_match["match_id"])

sample_events_df, sample_source = load_match_events(
    match_id=SAMPLE_MATCH_ID,
    cache_dir=EVENT_CACHE_DIR,
    overwrite_cache=CONFIG["OVERWRITE_CACHE"],
    max_retries=CONFIG["MAX_RETRIES"],
    request_delay_seconds=CONFIG["REQUEST_DELAY_SECONDS"],
)

assert not sample_events_df.empty, (
    "The sample event table is empty."
)

assert (
    sample_events_df["match_id"] == SAMPLE_MATCH_ID
).all(), "Incorrect parent match ID found in sample events."

assert sample_events_df["id"].notna().all(), (
    "Missing event IDs detected in sample events."
)

print("Single-match event validation passed.")
print(
    f"Fixture : "
    f"{sample_match['home_team']} vs "
    f"{sample_match['away_team']}"
)
print(f"Match ID: {SAMPLE_MATCH_ID}")
print(f"Source  : {sample_source}")
print(f"Events  : {len(sample_events_df):,}")
print(f"Columns : {len(sample_events_df.columns):,}")

display(
    sample_events_df[
        [
            "match_id",
            "id",
            "index",
            "period",
            "timestamp",
            "minute",
            "second",
            "type",
            "team",
        ]
    ].head(10)
)

Single-match event validation passed.
Fixture : Málaga vs Sevilla
Match ID: 3825562
Source  : api
Events  : 2,750
Columns : 92


,match_id,id,index,period,timestamp,minute,second,type,team
0,3825562,c412fb1b-8799-4189-878b-1616f75a346a,1,1,00:00:00.000,0,0,Starting XI,Málaga
1,3825562,9940ab8a-814e-4a56-b384-89979c290ff8,2,1,00:00:00.000,0,0,Starting XI,Sevilla
2,3825562,f5b49c4b-44c1-4ec8-a0e7-5483a692ba61,3,1,00:00:00.000,0,0,Half Start,Málaga
3,3825562,b7f29ea6-52fc-490f-bfef-c8ee60bf2d1c,4,1,00:00:00.000,0,0,Half Start,Sevilla
4,3825562,c1d7f0a1-7654-45d2-b64b-c6f8090cda83,5,1,00:00:01.428,0,1,Pass,Málaga
5,3825562,4da6fef9-3ee4-4b01-95d9-4bbef61769ed,6,1,00:00:02.179,0,2,Ball Receipt*,Málaga
6,3825562,c87534f8-7016-48c3-a4ed-5dd396496376,7,1,00:00:02.186,0,2,Pass,Málaga
7,3825562,19c5a461-96ed-4950-8c39-88953685f2a6,8,1,00:00:03.218,0,3,Ball Receipt*,Málaga
8,3825562,c3c3074d-a859-411b-bdbf-a3e1b2d0f0b9,9,1,00:00:05.106,0,5,Pass,Málaga
9,3825562,6b476a0e-1de8-4e0d-a914-37e36d332ded,10,1,00:00:06.808,0,6,Ball Receipt*,Málaga


In [6]:
# -------------------------------------------------------------------------
# SAMPLE EVENT-TYPE DISTRIBUTION
# -------------------------------------------------------------------------

sample_event_type_counts = (
    sample_events_df["type"]
    .value_counts(dropna=False)
    .rename_axis("event_type")
    .reset_index(name="event_count")
)

display(sample_event_type_counts.head(20))

,event_type,event_count
0,Pass,771
1,Ball Receipt*,708
2,Carry,524
3,Pressure,238
4,Ball Recovery,86
5,Duel,62
6,Goal Keeper,43
7,Shot,36
8,Foul Committed,33
9,Dribble,31


## Full-season event ingestion

The validated single-match loader is now applied to every fixture in the selected season.

For each match, the pipeline:

1. checks whether a cached Parquet file already exists;
2. loads the cached event table when available;
3. otherwise requests the event stream from StatsBomb;
4. validates the event schema and parent match identifier;
5. records the ingestion status;
6. continues even if an individual match fails.

A separate ingestion log is maintained so that failures can be inspected and retried without discarding successfully processed matches.

This design is intentionally fault tolerant. A temporary failure for one fixture should not force the entire season to be downloaded again.

In [7]:
# -------------------------------------------------------------------------
# FULL-SEASON EVENT INGESTION
# -------------------------------------------------------------------------

ingestion_records = []
season_event_tables = []

total_matches = len(matches_df)

print(f"Beginning event ingestion for {total_matches:,} matches.")
print("-" * 72)

for row_number, match_row in matches_df.iterrows():

    match_id = int(match_row["match_id"])
    home_team = str(match_row["home_team"])
    away_team = str(match_row["away_team"])
    match_date = pd.to_datetime(match_row["clean_date"])

    start_time = time.perf_counter()

    try:
        match_events_df, source = load_match_events(
            match_id=match_id,
            cache_dir=EVENT_CACHE_DIR,
            overwrite_cache=CONFIG["OVERWRITE_CACHE"],
            max_retries=CONFIG["MAX_RETRIES"],
            request_delay_seconds=CONFIG[
                "REQUEST_DELAY_SECONDS"
            ],
        )

        # Attach useful parent-match metadata.
        match_events_df = match_events_df.copy()

        match_events_df["match_date"] = match_date
        match_events_df["home_team"] = home_team
        match_events_df["away_team"] = away_team

        season_event_tables.append(match_events_df)

        elapsed_seconds = time.perf_counter() - start_time

        ingestion_records.append(
            {
                "match_id": match_id,
                "match_date": match_date,
                "home_team": home_team,
                "away_team": away_team,
                "status": "success",
                "source": source,
                "event_count": len(match_events_df),
                "column_count": len(match_events_df.columns),
                "elapsed_seconds": elapsed_seconds,
                "error_type": None,
                "error_message": None,
            }
        )

        print(
            f"[{row_number + 1:03d}/{total_matches:03d}] "
            f"SUCCESS | {home_team} vs {away_team} | "
            f"{len(match_events_df):,} events | "
            f"{source}"
        )

    except Exception as exc:
        elapsed_seconds = time.perf_counter() - start_time

        ingestion_records.append(
            {
                "match_id": match_id,
                "match_date": match_date,
                "home_team": home_team,
                "away_team": away_team,
                "status": "failed",
                "source": None,
                "event_count": 0,
                "column_count": 0,
                "elapsed_seconds": elapsed_seconds,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

        print(
            f"[{row_number + 1:03d}/{total_matches:03d}] "
            f"FAILED  | {home_team} vs {away_team} | "
            f"{type(exc).__name__}: {exc}"
        )

print("-" * 72)
print("Full-season ingestion loop finished.")

Beginning event ingestion for 380 matches.
------------------------------------------------------------------------
[001/380] SUCCESS | Málaga vs Sevilla | 2,750 events | cache
[002/380] SUCCESS | Atlético Madrid vs Las Palmas | 3,827 events | api
[003/380] SUCCESS | RC Deportivo La Coruña vs Real Sociedad | 3,279 events | api
[004/380] SUCCESS | Espanyol vs Getafe | 3,296 events | api
[005/380] SUCCESS | Rayo Vallecano vs Valencia | 3,289 events | api
[006/380] SUCCESS | Athletic Club vs Barcelona | 3,775 events | api
[007/380] SUCCESS | Sporting Gijón vs Real Madrid | 3,767 events | api
[008/380] SUCCESS | Real Betis vs Villarreal | 2,671 events | api
[009/380] SUCCESS | Levante UD vs Celta Vigo | 3,558 events | api
[010/380] SUCCESS | Granada vs Eibar | 3,236 events | api
[011/380] SUCCESS | Villarreal vs Espanyol | 3,307 events | api
[012/380] SUCCESS | Barcelona vs Málaga | 3,361 events | api
[013/380] SUCCESS | Real Madrid vs Real Betis | 3,430 events | api
[014/380] SUCCESS | Re

In [8]:
# -------------------------------------------------------------------------
# INGESTION LOG
# -------------------------------------------------------------------------

ingestion_log_df = pd.DataFrame(ingestion_records)

ingestion_log_df = ingestion_log_df.sort_values(
    by=["match_date", "match_id"]
).reset_index(drop=True)

ingestion_log_df.to_csv(
    INGESTION_LOG_PATH,
    index=False
)

successful_matches = (
    ingestion_log_df["status"] == "success"
).sum()

failed_matches = (
    ingestion_log_df["status"] == "failed"
).sum()

print("Ingestion log saved.")
print(f"Path              : {INGESTION_LOG_PATH.resolve()}")
print(f"Successful matches: {successful_matches:,}")
print(f"Failed matches    : {failed_matches:,}")
print(
    f"Coverage          : "
    f"{successful_matches / total_matches:.2%}"
)

display(
    ingestion_log_df.head()
)

Ingestion log saved.
Path              : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/event_ingestion_log_laliga_2015_2016.csv
Successful matches: 380
Failed matches    : 0
Coverage          : 100.00%


,match_id,match_date,home_team,away_team,status,source,event_count,column_count,elapsed_seconds,error_type,error_message
0,3825562,2015-08-21,Málaga,Sevilla,success,cache,2750,95,0.013130,None,None
1,3825563,2015-08-22,Atlético Madrid,Las Palmas,success,api,3827,92,1.817458,None,None
2,3825564,2015-08-22,RC Deportivo La Coruña,Real Sociedad,success,api,3279,90,1.629625,None,None
3,3825565,2015-08-22,Espanyol,Getafe,success,api,3296,92,2.390737,None,None
4,3825566,2015-08-22,Rayo Vallecano,Valencia,success,api,3289,91,1.523649,None,None


In [9]:
# -------------------------------------------------------------------------
# FAILURE INSPECTION
# -------------------------------------------------------------------------

failed_ingestion_df = ingestion_log_df[
    ingestion_log_df["status"] == "failed"
].copy()

if failed_ingestion_df.empty:
    print("No event-ingestion failures were detected.")
else:
    print(
        f"{len(failed_ingestion_df):,} match(es) failed ingestion."
    )

    display(
        failed_ingestion_df[
            [
                "match_id",
                "match_date",
                "home_team",
                "away_team",
                "error_type",
                "error_message",
            ]
        ]
    )

No event-ingestion failures were detected.


In [11]:
# -------------------------------------------------------------------------
# CONCATENATE FULL-SEASON EVENT TABLE
# -------------------------------------------------------------------------

if not season_event_tables:
    raise RuntimeError(
        "No event tables were successfully ingested."
    )

season_events_df = pd.concat(
    season_event_tables,
    ignore_index=True,
    sort=False
)

season_events_df = season_events_df.sort_values(
    by=["match_date", "match_id", "period", "index"]
).reset_index(drop=True)

print("Successful event tables concatenated.")
print(f"Total events : {len(season_events_df):,}")
print(
    f"Unique matches: "
    f"{season_events_df['match_id'].nunique():,}"
)
print(f"Columns      : {len(season_events_df.columns):,}")

Successful event tables concatenated.
Total events : 1,295,354
Unique matches: 380
Columns      : 120


In [12]:
# -------------------------------------------------------------------------
# FULL-SEASON VALIDATION
# -------------------------------------------------------------------------

successful_match_ids = set(
    ingestion_log_df.loc[
        ingestion_log_df["status"] == "success",
        "match_id",
    ].astype(int)
)

event_match_ids = set(
    season_events_df["match_id"].astype(int)
)

assert event_match_ids == successful_match_ids, (
    "Mismatch between successful ingestion-log matches "
    "and event-table matches."
)

assert season_events_df["match_id"].notna().all(), (
    "Missing match IDs detected in the full event table."
)

assert season_events_df["id"].notna().all(), (
    "Missing event IDs detected."
)

assert season_events_df["index"].notna().all(), (
    "Missing event ordering indices detected."
)

duplicate_event_ids = (
    season_events_df["id"]
    .duplicated()
    .sum()
)

if duplicate_event_ids > 0:
    raise ValueError(
        f"Detected {duplicate_event_ids:,} duplicate event IDs."
    )

events_per_match_df = (
    season_events_df
    .groupby("match_id")
    .size()
    .rename("event_count")
    .reset_index()
)

assert (events_per_match_df["event_count"] > 0).all(), (
    "At least one match has no events."
)

print("Full-season event validation passed.")
print(
    f"Validated matches: "
    f"{len(events_per_match_df):,}"
)
print(
    f"Minimum events in one match: "
    f"{events_per_match_df['event_count'].min():,}"
)
print(
    f"Maximum events in one match: "
    f"{events_per_match_df['event_count'].max():,}"
)
print(
    f"Median events per match: "
    f"{events_per_match_df['event_count'].median():,.0f}"
)

Full-season event validation passed.
Validated matches: 380
Minimum events in one match: 2,614
Maximum events in one match: 4,326
Median events per match: 3,375


In [13]:
# -------------------------------------------------------------------------
# CHECK PARQUET-SAFE COLUMN TYPES
# -------------------------------------------------------------------------

nested_object_columns = []

for column in season_events_df.columns:

    if season_events_df[column].dtype != "object":
        continue

    non_null_values = season_events_df[column].dropna()

    contains_nested = non_null_values.map(
        lambda value: isinstance(
            value,
            (dict, list, tuple, set, np.ndarray)
        )
    ).any()

    if contains_nested:
        nested_object_columns.append(column)

print(
    f"Nested object columns detected: "
    f"{len(nested_object_columns)}"
)

if nested_object_columns:
    print("\nNested columns:")
    for column in nested_object_columns:
        print(f"  - {column}")
else:
    print("No nested object columns were detected.")

Nested object columns detected: 9

Nested columns:
  - carry_end_location
  - goalkeeper_end_location
  - location
  - pass_end_location
  - related_events
  - shot_end_location
  - shot_freeze_frame
  - tactics
  - 50_50


In [14]:
# -------------------------------------------------------------------------
# INSPECT NESTED EVENT COLUMNS
# -------------------------------------------------------------------------

nested_column_examples = {}

for column in nested_object_columns:
    example_values = (
        season_events_df[column]
        .dropna()
        .head(3)
        .tolist()
    )

    nested_column_examples[column] = example_values

for column, examples in nested_column_examples.items():
    print(f"\n{column}")
    print("-" * len(column))

    for example in examples:
        print(repr(example))


carry_end_location
------------------
array([38.1, 11. ])
array([35.4, 52.6])
array([62.2, 58.5])

goalkeeper_end_location
-----------------------
array([ 2.4, 43.2])
array([ 2.7, 36.6])
array([ 1.5, 40.3])

location
--------
array([60., 40.])
array([59.4, 42.2])
array([59.4, 39.4])

pass_end_location
-----------------
array([59.4, 42.2])
array([41.8, 36.1])
array([40.8,  6.6])

related_events
--------------
array(['b7f29ea6-52fc-490f-bfef-c8ee60bf2d1c'], dtype=object)
array(['f5b49c4b-44c1-4ec8-a0e7-5483a692ba61'], dtype=object)
array(['4da6fef9-3ee4-4b01-95d9-4bbef61769ed'], dtype=object)

shot_end_location
-----------------
array([108.7,  21.5])
array([117.3,  38.7,   0.5])
array([117.3,  43.5,   0.9])

shot_freeze_frame
-----------------
array([{'location': array([106.1,  35.4]), 'player': {'id': 3497, 'name': 'Vicente Iborra De La Fuente'}, 'position': {'id': 9, 'name': 'Right Defensive Midfield'}, 'teammate': False},
       {'location': array([104.4,  24.7]), 'player': {'id': 23

In [15]:
# -------------------------------------------------------------------------
# COORDINATE EXTRACTION HELPERS
# -------------------------------------------------------------------------

def extract_coordinate(value, coordinate_index):
    """
    Extract one coordinate from a StatsBomb location-style value.

    Parameters
    ----------
    value : object
        Expected to be a list, tuple or NumPy array.

    coordinate_index : int
        Position of the coordinate to extract:
        0 for x, 1 for y and 2 for z.

    Returns
    -------
    float or np.nan
        Extracted coordinate, or NaN when unavailable.
    """

    if isinstance(value, (list, tuple, np.ndarray)):
        if len(value) > coordinate_index:
            coordinate = value[coordinate_index]

            if coordinate is not None:
                try:
                    return float(coordinate)
                except (TypeError, ValueError):
                    return np.nan

    return np.nan

In [16]:
# -------------------------------------------------------------------------
# BUILD ANALYSIS-READY FULL-SEASON EVENT TABLE
# -------------------------------------------------------------------------

analysis_events_df = season_events_df.copy()


# -------------------------------------------------------------------------
# EXPAND EVENT START LOCATION
# -------------------------------------------------------------------------

analysis_events_df["location_x"] = (
    analysis_events_df["location"]
    .map(lambda value: extract_coordinate(value, 0))
)

analysis_events_df["location_y"] = (
    analysis_events_df["location"]
    .map(lambda value: extract_coordinate(value, 1))
)


# -------------------------------------------------------------------------
# EXPAND PASS END LOCATION
# -------------------------------------------------------------------------

analysis_events_df["pass_end_x"] = (
    analysis_events_df["pass_end_location"]
    .map(lambda value: extract_coordinate(value, 0))
)

analysis_events_df["pass_end_y"] = (
    analysis_events_df["pass_end_location"]
    .map(lambda value: extract_coordinate(value, 1))
)


# -------------------------------------------------------------------------
# EXPAND CARRY END LOCATION
# -------------------------------------------------------------------------

analysis_events_df["carry_end_x"] = (
    analysis_events_df["carry_end_location"]
    .map(lambda value: extract_coordinate(value, 0))
)

analysis_events_df["carry_end_y"] = (
    analysis_events_df["carry_end_location"]
    .map(lambda value: extract_coordinate(value, 1))
)


# -------------------------------------------------------------------------
# EXPAND SHOT END LOCATION
# -------------------------------------------------------------------------

analysis_events_df["shot_end_x"] = (
    analysis_events_df["shot_end_location"]
    .map(lambda value: extract_coordinate(value, 0))
)

analysis_events_df["shot_end_y"] = (
    analysis_events_df["shot_end_location"]
    .map(lambda value: extract_coordinate(value, 1))
)

analysis_events_df["shot_end_z"] = (
    analysis_events_df["shot_end_location"]
    .map(lambda value: extract_coordinate(value, 2))
)


# -------------------------------------------------------------------------
# EXPAND GOALKEEPER END LOCATION
# -------------------------------------------------------------------------

analysis_events_df["goalkeeper_end_x"] = (
    analysis_events_df["goalkeeper_end_location"]
    .map(lambda value: extract_coordinate(value, 0))
)

analysis_events_df["goalkeeper_end_y"] = (
    analysis_events_df["goalkeeper_end_location"]
    .map(lambda value: extract_coordinate(value, 1))
)

print("Nested coordinate columns expanded successfully.")

Nested coordinate columns expanded successfully.


In [17]:
# -------------------------------------------------------------------------
# CREATE CONTINUOUS MATCH-TIME VARIABLES
# -------------------------------------------------------------------------

analysis_events_df["minute"] = pd.to_numeric(
    analysis_events_df["minute"],
    errors="coerce"
)

analysis_events_df["second"] = pd.to_numeric(
    analysis_events_df["second"],
    errors="coerce"
)

analysis_events_df["event_seconds"] = (
    analysis_events_df["minute"] * 60
    + analysis_events_df["second"]
)

analysis_events_df["event_minute_decimal"] = (
    analysis_events_df["event_seconds"] / 60
)

assert analysis_events_df["event_seconds"].notna().all(), (
    "Some events have invalid minute or second values."
)

assert (analysis_events_df["event_seconds"] >= 0).all(), (
    "Negative event times were detected."
)

print("Continuous event-time variables created.")
print(
    f"Maximum recorded match minute: "
    f"{analysis_events_df['event_minute_decimal'].max():.2f}"
)

Continuous event-time variables created.
Maximum recorded match minute: 97.07


In [18]:
# -------------------------------------------------------------------------
# SELECT ANALYSIS-READY EVENT COLUMNS
# -------------------------------------------------------------------------

preferred_event_columns = [
    # Relational identity
    "match_id",
    "match_date",
    "home_team",
    "away_team",

    # Event identity and ordering
    "id",
    "index",
    "period",
    "timestamp",
    "minute",
    "second",
    "event_seconds",
    "event_minute_decimal",

    # Event ownership and context
    "type",
    "team",
    "team_id",
    "player",
    "player_id",
    "position",
    "possession",
    "possession_team",
    "possession_team_id",
    "play_pattern",
    "under_pressure",
    "counterpress",
    "duration",

    # Starting coordinates
    "location_x",
    "location_y",

    # Pass information
    "pass_recipient",
    "pass_recipient_id",
    "pass_length",
    "pass_angle",
    "pass_height",
    "pass_type",
    "pass_outcome",
    "pass_body_part",
    "pass_cross",
    "pass_cut_back",
    "pass_switch",
    "pass_through_ball",
    "pass_shot_assist",
    "pass_goal_assist",
    "pass_end_x",
    "pass_end_y",

    # Carry information
    "carry_end_x",
    "carry_end_y",

    # Shot information
    "shot_statsbomb_xg",
    "shot_outcome",
    "shot_type",
    "shot_body_part",
    "shot_technique",
    "shot_first_time",
    "shot_one_on_one",
    "shot_end_x",
    "shot_end_y",
    "shot_end_z",

    # Defensive and duel information
    "duel_type",
    "duel_outcome",
    "interception_outcome",
    "ball_recovery_recovery_failure",
    "clearance_body_part",
    "block_deflection",
    "dribble_outcome",
    "foul_committed_card",
    "foul_won_defensive",

    # Goalkeeper information
    "goalkeeper_type",
    "goalkeeper_outcome",
    "goalkeeper_position",
    "goalkeeper_technique",
    "goalkeeper_body_part",
    "goalkeeper_end_x",
    "goalkeeper_end_y",

    # Substitutions and cards
    "substitution_replacement",
    "substitution_outcome",
    "bad_behaviour_card",
]

available_event_columns = [
    column
    for column in preferred_event_columns
    if column in analysis_events_df.columns
]

missing_optional_columns = [
    column
    for column in preferred_event_columns
    if column not in analysis_events_df.columns
]

flat_events_df = analysis_events_df[
    available_event_columns
].copy()

print("Analysis-ready event table created.")
print(f"Rows             : {len(flat_events_df):,}")
print(f"Selected columns : {len(flat_events_df.columns):,}")
print(f"Optional missing : {len(missing_optional_columns):,}")

if missing_optional_columns:
    print("\nOptional columns not present in this season:")
    for column in missing_optional_columns:
        print(f"  - {column}")

Analysis-ready event table created.
Rows             : 1,295,354
Selected columns : 74
Optional missing : 0


In [19]:
# -------------------------------------------------------------------------
# VALIDATE FLAT EVENT SCHEMA
# -------------------------------------------------------------------------

remaining_nested_columns = []

for column in flat_events_df.columns:

    if flat_events_df[column].dtype != "object":
        continue

    non_null_values = flat_events_df[column].dropna()

    contains_nested = non_null_values.map(
        lambda value: isinstance(
            value,
            (dict, list, tuple, set, np.ndarray)
        )
    ).any()

    if contains_nested:
        remaining_nested_columns.append(column)

if remaining_nested_columns:
    raise ValueError(
        "Nested values remain in the analysis-ready event table: "
        f"{remaining_nested_columns}"
    )

print("Flat-schema validation passed.")
print("No nested list or dictionary values remain.")

Flat-schema validation passed.
No nested list or dictionary values remain.


In [20]:
# -------------------------------------------------------------------------
# STANDARDISE KEY EVENT DATA TYPES
# -------------------------------------------------------------------------

integer_columns = [
    "match_id",
    "index",
    "period",
    "minute",
    "second",
]

for column in integer_columns:
    if column in flat_events_df.columns:
        flat_events_df[column] = pd.to_numeric(
            flat_events_df[column],
            errors="raise"
        ).astype("int64")

nullable_integer_columns = [
    "team_id",
    "player_id",
    "possession",
    "possession_team_id",
    "pass_recipient_id",
]

for column in nullable_integer_columns:
    if column in flat_events_df.columns:
        flat_events_df[column] = pd.to_numeric(
            flat_events_df[column],
            errors="coerce"
        ).astype("Int64")

float_columns = [
    "event_seconds",
    "event_minute_decimal",
    "duration",
    "location_x",
    "location_y",
    "pass_length",
    "pass_angle",
    "pass_end_x",
    "pass_end_y",
    "carry_end_x",
    "carry_end_y",
    "shot_statsbomb_xg",
    "shot_end_x",
    "shot_end_y",
    "shot_end_z",
    "goalkeeper_end_x",
    "goalkeeper_end_y",
]

for column in float_columns:
    if column in flat_events_df.columns:
        flat_events_df[column] = pd.to_numeric(
            flat_events_df[column],
            errors="coerce"
        ).astype("float64")

flat_events_df["match_date"] = pd.to_datetime(
    flat_events_df["match_date"],
    errors="raise"
)

print("Key data types standardised.")

Key data types standardised.


In [22]:
# -------------------------------------------------------------------------
# SPATIAL COORDINATE VALIDATION AND BOUNDARY CORRECTION
# -------------------------------------------------------------------------

# StatsBomb's nominal pitch dimensions.
PITCH_X_MIN = 0.0
PITCH_X_MAX = 120.0
PITCH_Y_MIN = 0.0
PITCH_Y_MAX = 80.0

# Allow a small tolerance for provider-side annotation or rounding.
# Values beyond these limits are treated as genuinely invalid.
COORDINATE_TOLERANCE = 1.0

xy_coordinate_pairs = [
    ("location_x", "location_y"),
    ("pass_end_x", "pass_end_y"),
    ("carry_end_x", "carry_end_y"),
    ("shot_end_x", "shot_end_y"),
    ("goalkeeper_end_x", "goalkeeper_end_y"),
]

coordinate_validation_records = []
minor_boundary_rows = []
serious_invalid_rows = []

for x_column, y_column in xy_coordinate_pairs:

    if (
        x_column not in flat_events_df.columns
        or y_column not in flat_events_df.columns
    ):
        continue

    valid_mask = (
        flat_events_df[x_column].notna()
        & flat_events_df[y_column].notna()
    )

    valid_rows = flat_events_df.loc[
        valid_mask,
        [
            "match_id",
            "id",
            "type",
            x_column,
            y_column,
        ],
    ].copy()

    if valid_rows.empty:
        continue

    # ---------------------------------------------------------------------
    # VALUES OUTSIDE THE NOMINAL 120 x 80 RANGE
    # ---------------------------------------------------------------------

    outside_nominal_mask = (
        (valid_rows[x_column] < PITCH_X_MIN)
        | (valid_rows[x_column] > PITCH_X_MAX)
        | (valid_rows[y_column] < PITCH_Y_MIN)
        | (valid_rows[y_column] > PITCH_Y_MAX)
    )

    # ---------------------------------------------------------------------
    # VALUES OUTSIDE THE ACCEPTABLE TOLERANCE
    # ---------------------------------------------------------------------

    serious_invalid_mask = (
        (valid_rows[x_column] < PITCH_X_MIN - COORDINATE_TOLERANCE)
        | (valid_rows[x_column] > PITCH_X_MAX + COORDINATE_TOLERANCE)
        | (valid_rows[y_column] < PITCH_Y_MIN - COORDINATE_TOLERANCE)
        | (valid_rows[y_column] > PITCH_Y_MAX + COORDINATE_TOLERANCE)
    )

    minor_overflow_mask = (
        outside_nominal_mask
        & ~serious_invalid_mask
    )

    if minor_overflow_mask.any():
        minor_rows = valid_rows.loc[
            minor_overflow_mask
        ].copy()

        minor_rows["coordinate_pair"] = (
            f"{x_column}, {y_column}"
        )

        minor_boundary_rows.append(minor_rows)

    if serious_invalid_mask.any():
        invalid_rows = valid_rows.loc[
            serious_invalid_mask
        ].copy()

        invalid_rows["coordinate_pair"] = (
            f"{x_column}, {y_column}"
        )

        serious_invalid_rows.append(invalid_rows)

    coordinate_validation_records.append(
        {
            "coordinate_pair": f"{x_column}, {y_column}",
            "available_rows": len(valid_rows),
            "outside_nominal_range": int(
                outside_nominal_mask.sum()
            ),
            "minor_boundary_overflow": int(
                minor_overflow_mask.sum()
            ),
            "serious_invalid_values": int(
                serious_invalid_mask.sum()
            ),
            "minimum_x": valid_rows[x_column].min(),
            "maximum_x": valid_rows[x_column].max(),
            "minimum_y": valid_rows[y_column].min(),
            "maximum_y": valid_rows[y_column].max(),
        }
    )


coordinate_validation_df = pd.DataFrame(
    coordinate_validation_records
)

display(coordinate_validation_df)


# -------------------------------------------------------------------------
# REJECT SERIOUSLY INVALID COORDINATES
# -------------------------------------------------------------------------

if serious_invalid_rows:
    serious_invalid_coordinates_df = pd.concat(
        serious_invalid_rows,
        ignore_index=True,
    )

    display(serious_invalid_coordinates_df.head(20))

    raise ValueError(
        "Coordinates beyond the permitted tolerance were detected. "
        f"Rows affected: {len(serious_invalid_coordinates_df):,}"
    )

print(
    "No seriously invalid spatial coordinates were detected."
)


# -------------------------------------------------------------------------
# REPORT MINOR BOUNDARY OVERFLOW
# -------------------------------------------------------------------------

if minor_boundary_rows:
    minor_boundary_coordinates_df = pd.concat(
        minor_boundary_rows,
        ignore_index=True,
    )

    print(
        "Minor boundary overflows detected: "
        f"{len(minor_boundary_coordinates_df):,}"
    )

    display(
        minor_boundary_coordinates_df.head(20)
    )
else:
    minor_boundary_coordinates_df = pd.DataFrame()

    print("No minor coordinate overflows were detected.")


# -------------------------------------------------------------------------
# CLIP MINOR BOUNDARY VALUES TO THE NOMINAL PITCH
# -------------------------------------------------------------------------

for x_column, y_column in xy_coordinate_pairs:

    if x_column in flat_events_df.columns:
        flat_events_df[x_column] = (
            flat_events_df[x_column]
            .clip(
                lower=PITCH_X_MIN,
                upper=PITCH_X_MAX,
            )
        )

    if y_column in flat_events_df.columns:
        flat_events_df[y_column] = (
            flat_events_df[y_column]
            .clip(
                lower=PITCH_Y_MIN,
                upper=PITCH_Y_MAX,
            )
        )


# -------------------------------------------------------------------------
# FINAL POST-CORRECTION ASSERTIONS
# -------------------------------------------------------------------------

for x_column, y_column in xy_coordinate_pairs:

    if (
        x_column not in flat_events_df.columns
        or y_column not in flat_events_df.columns
    ):
        continue

    assert flat_events_df[x_column].dropna().between(
        PITCH_X_MIN,
        PITCH_X_MAX,
    ).all(), (
        f"Invalid values remain in {x_column}."
    )

    assert flat_events_df[y_column].dropna().between(
        PITCH_Y_MIN,
        PITCH_Y_MAX,
    ).all(), (
        f"Invalid values remain in {y_column}."
    )

print(
    "Spatial-coordinate validation and boundary correction passed."
)

,coordinate_pair,available_rows,outside_nominal_range,minor_boundary_overflow,serious_invalid_values,minimum_x,maximum_x,minimum_y,maximum_y
0,"location_x, location_y",1286172,9,9,0,0.1,120.3,0.1,80.4
1,"pass_end_x, pass_end_y",365288,16,16,0,0.1,120.9,0.1,80.7
2,"carry_end_x, carry_end_y",276171,1,1,0,0.1,120.2,0.1,80.0
3,"shot_end_x, shot_end_y",9168,0,0,0,70.5,120.0,0.1,80.0
4,"goalkeeper_end_x, goalkeeper_end_y",5861,0,0,0,0.1,40.8,11.5,60.4


No seriously invalid spatial coordinates were detected.
Minor boundary overflows detected: 26


,match_id,id,type,location_x,location_y,coordinate_pair,pass_end_x,pass_end_y,carry_end_x,carry_end_y
0,266467,45c20e6b-a085-4dd7-956d-c1247384960c,Ball Receipt*,120.3,59.8,"location_x, location_y",NaN,NaN,NaN,NaN
1,266467,ce19bf2c-0fb9-4a0d-b9e4-d883a7b2176b,Miscontrol,120.3,59.8,"location_x, location_y",NaN,NaN,NaN,NaN
2,266961,f554cda6-34f9-4d7f-82fe-09c1c2c53d81,Pass,120.2,23.8,"location_x, location_y",NaN,NaN,NaN,NaN
3,267273,d9c8a082-0c3c-432b-9af4-0cd30eccfd8e,Ball Receipt*,59.3,80.4,"location_x, location_y",NaN,NaN,NaN,NaN
4,267273,24b084f6-72f1-4310-90bd-22b8bca0fd68,Carry,59.3,80.4,"location_x, location_y",NaN,NaN,NaN,NaN
5,267273,e86a33c9-b796-46bb-9d17-0ff851044ed1,Ball Receipt*,120.1,75.8,"location_x, location_y",NaN,NaN,NaN,NaN
6,267273,40bbc451-1502-4c81-a0f8-17e287611050,Carry,120.1,75.8,"location_x, location_y",NaN,NaN,NaN,NaN
7,266106,1230c2ea-ac86-44df-98bd-307870af2c93,Ball Recovery,61.0,80.4,"location_x, location_y",NaN,NaN,NaN,NaN
8,266106,6f694afe-cc47-4193-8e7e-0de18ad9c51b,Carry,61.0,80.4,"location_x, location_y",NaN,NaN,NaN,NaN
9,266467,ae16d839-31cf-4d22-bc50-00f1bd482c3d,Pass,NaN,NaN,"pass_end_x, pass_end_y",29.4,80.4,NaN,NaN


Spatial-coordinate validation and boundary correction passed.


In [23]:
# -------------------------------------------------------------------------
# EXPORT ANALYSIS-READY FULL-SEASON EVENTS
# -------------------------------------------------------------------------

flat_events_df = flat_events_df.sort_values(
    by=["match_date", "match_id", "period", "index"]
).reset_index(drop=True)

flat_events_df.to_parquet(
    EVENT_OUTPUT_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print("Analysis-ready event table exported successfully.")
print(f"Path   : {EVENT_OUTPUT_PATH.resolve()}")
print(f"Rows   : {len(flat_events_df):,}")
print(f"Columns: {len(flat_events_df.columns):,}")

Analysis-ready event table exported successfully.
Path   : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/events_laliga_2015_2016.parquet
Rows   : 1,295,354
Columns: 74


In [24]:
# -------------------------------------------------------------------------
# RELOAD AND VALIDATE SAVED EVENT TABLE
# -------------------------------------------------------------------------

reloaded_events_df = pd.read_parquet(
    EVENT_OUTPUT_PATH
)

assert len(reloaded_events_df) == len(flat_events_df), (
    "Row count changed during export and reload."
)

assert reloaded_events_df["match_id"].nunique() == 380, (
    "The reloaded event table does not contain all 380 matches."
)

assert reloaded_events_df["id"].notna().all(), (
    "Missing event IDs detected after reload."
)

assert not reloaded_events_df["id"].duplicated().any(), (
    "Duplicate event IDs detected after reload."
)

assert set(reloaded_events_df["match_id"]) == set(
    matches_df["match_id"]
), "Reloaded events do not cover the complete match table."

print("Reload validation passed.")
print(f"Events validated : {len(reloaded_events_df):,}")
print(
    f"Matches validated: "
    f"{reloaded_events_df['match_id'].nunique():,}"
)

Reload validation passed.
Events validated : 1,295,354
Matches validated: 380


## Full-season ingestion summary

The complete La Liga 2015/16 event stream was successfully ingested and validated.

### Results

- **Matches requested:** 380
- **Matches successfully ingested:** 380
- **Failed matches:** 0
- **Coverage:** 100%
- **Total events:** 1,295,354
- **Minimum events per match:** 2,614
- **Maximum events per match:** 4,326
- **Median events per match:** 3,375

Each event remains connected to its parent match through the StatsBomb `match_id`. Match date, home-team identity and away-team identity were also attached to the event records.

The original event tables contained nested spatial and tactical structures. Rather than discarding the raw information, the pipeline retained the detailed per-match cache files and created a separate analysis-ready table. Spatial lists were expanded into explicit coordinate columns, while deeply nested objects such as freeze frames and tactics remain available through the raw cache.

The processed full-season event table now supports:

1. pass-network construction;
2. the multilayer-network paper adaptation;
3. rolling event-derived pre-match features;
4. player and lineup summaries;
5. complete in-play snapshot generation.

The next notebook will transform completed-match event streams into team-level spatial multilayer-network metrics. Those match-level metrics will later be aggregated over historical fixtures to produce temporally safe pre-match features.